### Supplement: 2-Structured Outputs with Pydantic Deep Dive

This supplement notebook breaks down `2-structured.py` cell by cell. It covers **Structured Outputs** using Pydantic and the OpenAI Python SDK:

1. **What is Pydantic and `BaseModel`?**: Defining data schemas with strict types.
2. **Schema Generation**: How `CalendarEvent` is converted to a JSON Schema under the hood.
3. **`create` vs. `parse`**: Understanding `client.chat.completions.parse(...)`.
4. **`message.content` vs. `message.parsed`**:
   - `message.content`: The raw JSON string returned by the LLM.
   - `message.parsed`: The deserialized, validated Pydantic object!
5. **Accessing Typed Attributes & Converting to Python Dicts**: Dot-notation vs. `.model_dump()`.
6. **Cheat Sheet**: `model_json_schema()` vs. `model_dump()` vs. `json.loads()` vs. `.parsed`.

---

##### Architectural Flow:
```
1. Define Pydantic Schema:
   class CalendarEvent(BaseModel):
       name: str
       date: str
       participants: list[str]
            │
            ▼
2. SDK calls .model_json_schema(), tightens it for strict mode,
   and puts it in the request body sent to OpenAI
            │
            ▼
3. OpenAI API uses Constrained Sampling (the emitted JSON is
   guaranteed to match the schema — unless the model refuses,
   or generation is cut off by a token limit)
            │
            ▼
4. Response arrives back at SDK:
   ├── message.content -> raw JSON string: '{"name": "...", "date": "...", ...}'
   └── message.parsed  -> CalendarEvent(name='...', date='...', participants=[...])
```

> **Note on `.beta.`**: older tutorials call `client.beta.chat.completions.parse(...)`. Structured-output parsing has since graduated out of beta, so this notebook uses `client.chat.completions.parse(...)` — matching `2-structured.py`. The `.beta.` path still exists in the installed SDK (openai v2.1.0) and is functionally equivalent, so older code keeps working. The one visible difference is cosmetic: `.beta.` prints its type as `ParsedChatCompletion[CalendarEvent]`, while the stable path prints `ParsedChatCompletion[TypeVar]` (see Section 3). Prefer the non-`beta` path in anything new.

#### 1. Imports and Environment Setup

Notice we import:
- `OpenAI`: The API client class.
- `BaseModel`, `Field`: From `pydantic`. `BaseModel` is the base class for defining data contracts and schemas.

In [2]:
import os
import json
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

# Load API key
load_dotenv(find_dotenv(usecwd=True))
if not os.getenv("OPENAI_API_KEY"):
    load_dotenv(r"C:\Users\ashut\ML_Practice\LLM_Learning_Sandbox\.env")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("Client initialized successfully.")
print("Pydantic BaseModel class:", BaseModel)


Client initialized successfully.
Pydantic BaseModel class: <class 'pydantic.main.BaseModel'>


#### 2. Defining the Schema Class: `CalendarEvent(BaseModel)`

##### What is `BaseModel`?
`BaseModel` is Pydantic's core class. When you subclass `BaseModel`:
- It parses and validates data according to type hints (`str`, `list[str]`, etc.).
- It can automatically export a standard JSON Schema via `.model_json_schema()`.
- It allows serialization back to dicts via `.model_dump()`.

Below we call `.model_json_schema()` **purely for inspection**, so you can see the shape the SDK will derive from your class. Note that nothing in this notebook sends that `schema` variable anywhere — the SDK regenerates it internally when you pass `response_format=CalendarEvent` in Section 3. The next cell explains exactly who consumes it.

In [3]:
class CalendarEvent(BaseModel):
    name: str = Field(description="The name or title of the event")
    date: str = Field(description="The date or day of the event")
    participants: list[str] = Field(description="List of attendee names")

print("Class name:", CalendarEvent.__name__)
print("Inherits from:", [b.__name__ for b in CalendarEvent.__bases__])
print("Declared fields:", list(CalendarEvent.model_fields.keys()))

print("\n--- JSON Schema generated by Pydantic (Sent to OpenAI API under the hood) ---")
schema = CalendarEvent.model_json_schema()
print(json.dumps(schema, indent=2))


Class name: CalendarEvent
Inherits from: ['BaseModel']
Declared fields: ['name', 'date', 'participants']

--- JSON Schema generated by Pydantic (Sent to OpenAI API under the hood) ---
{
  "properties": {
    "name": {
      "description": "The name or title of the event",
      "title": "Name",
      "type": "string"
    },
    "date": {
      "description": "The date or day of the event",
      "title": "Date",
      "type": "string"
    },
    "participants": {
      "description": "List of attendee names",
      "items": {
        "type": "string"
      },
      "title": "Participants",
      "type": "array"
    }
  },
  "required": [
    "name",
    "date",
    "participants"
  ],
  "title": "CalendarEvent",
  "type": "object"
}


##### Wait — who actually *uses* this JSON Schema, and for what?

This is the single most confusing part of structured outputs, so let's be precise.

**You never call `.model_json_schema()` yourself in normal usage.** The cell above is a teaching X-ray. The schema's real consumer is **OpenAI's inference server**, not your Python code.

Here is the actual chain of custody:

1. You pass the **class itself** (not a schema) to the SDK: `response_format=CalendarEvent`.
2. **The SDK** — inside `openai/lib/_parsing/_completions.py` — calls `CalendarEvent.model_json_schema()` for you.
3. **The SDK tightens it for strict mode**: it recursively injects `"additionalProperties": false` into every object and wraps the result with a `name` and `strict: true`.
4. That payload goes over the wire in the request body.
5. **OpenAI's server** compiles the schema into a grammar and uses it for **constrained decoding** — at each step, tokens that would break the schema are masked out of the sampling distribution. The model is *mechanically unable* to emit a wrong field name, a wrong type, or a missing required field.

So the answer to *"where is this JSON used, by whom, for what?"* is: **by OpenAI's token sampler, to make invalid output impossible.** It is a contract shipped to the model, not data for your program.

##### Important: the raw Pydantic schema is *not* byte-for-byte what gets sent

The printout above is Pydantic's generic export. Compare it to what the SDK actually transmits:

```python
from openai.lib._parsing._completions import type_to_response_format_param
print(json.dumps(type_to_response_format_param(CalendarEvent), indent=2))
```

```jsonc
{
  "type": "json_schema",
  "json_schema": {
    "schema": {
      "properties": { /* ...same as above... */ },
      "required": ["name", "date", "participants"],
      "title": "CalendarEvent",
      "type": "object",
      "additionalProperties": false   // <-- ADDED by the SDK for strict mode
    },
    "name": "CalendarEvent",          // <-- ADDED
    "strict": true                    // <-- ADDED
  }
}
```

Two practical consequences:
- The `description=` text you wrote in each `Field(...)` **does** survive into the schema, so it reaches the model and acts as a per-field prompt. Descriptive `Field` descriptions genuinely improve extraction quality.
- Strict mode forbids optional/extra keys. Every field is required, which is why you model "maybe missing" as `Optional[str]` (i.e. `str | None`) rather than by omitting the field.

#### 3. The API Call: `client.chat.completions.parse(...)`

##### Why `parse(...)` instead of `create(...)`?
- **`create(...)`**: You get back a standard `ChatCompletion`. `message.content` is just a `str`, and `message.parsed` does not exist. To get an object you must deserialize it yourself:
  ```python
  data = json.loads(completion.choices[0].message.content)  # -> plain dict
  event = CalendarEvent(**data)                             # -> validate by hand
  ```
- **`parse(...)`**: A high-level helper in the OpenAI SDK that:
  1. Converts your Pydantic class to a JSON Schema and sends it with `strict: true`.
  2. The model generates strictly conforming JSON tokens via constrained decoding.
  3. The SDK automatically validates the JSON and instantiates your `CalendarEvent` class!
  4. The instantiated object is placed in `completion.choices[0].message.parsed`.

> **A precise distinction.** `create()` is not inherently "unsafe JSON" — it also accepts `response_format={"type": "json_schema", ...}` with strict mode, giving the same generation guarantee. The catch is that you must hand-write that schema dict and do your own `json.loads()` + validation. So `parse()` is not buying you *reliability the API otherwise lacks*; it is buying you **the Pydantic-class-to-schema conversion on the way out, and the deserialization on the way back.**
>
> The older "JSON mode" (`response_format={"type": "json_object"}`) is the genuinely weaker option: it guarantees only *syntactically valid* JSON, with no control over which fields appear. That is the case where "hope the model didn't invent fields" actually applies.

##### Don't be thrown by `ParsedChatCompletion[TypeVar]` in the output below

The printed type reads `ParsedChatCompletion[TypeVar]` rather than the more informative `ParsedChatCompletion[CalendarEvent]` you'd see from the older `.beta.` path. This is **purely cosmetic** — the non-beta `parse()` doesn't bind the generic parameter into the runtime repr.

Nothing about the behaviour changes, as Section 4 confirms: `message.parsed` is a genuine `CalendarEvent` and `isinstance(message.parsed, CalendarEvent)` is `True`. Static type checkers still infer the correct type in your editor; only this one `print(type(...))` string is less specific.

In [4]:
prompt_messages = [
    {"role": "system", "content": "Extract the event information."},
    {
        "role": "user",
        "content": "Alice and Bob are going to a science fair on Friday.",
    },
]

completion = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=prompt_messages,
    response_format=CalendarEvent,
)

print("Parsed completion call successful!")
print("Completion object type:", type(completion))
print("Choices length:", len(completion.choices))

Parsed completion call successful!
Completion object type: <class 'openai.types.chat.parsed_chat_completion.ParsedChatCompletion[TypeVar]'>
Choices length: 1


#### 4. Comparing `message.content` vs. `message.parsed`

This is the most critical distinction in Structured Outputs:
1. **`message.content`**: Contains the **raw JSON string** sent across the wire by the LLM.
2. **`message.parsed`**: Contains the **instantiated Python Pydantic object** (`CalendarEvent`).

Both are populated on a successful call — `parsed` is simply the SDK having already done `json.loads()` + Pydantic validation on `content` for you.

The cell also prints **`message.refusal`**, which is the escape hatch in the schema guarantee. If the model declines the request on safety grounds, it returns a plain-text refusal *instead of* schema-conforming JSON: `refusal` holds that text, and `parsed` is `None`. In production this is what you branch on:

```python
if message.refusal:
    handle_refusal(message.refusal)
else:
    event = message.parsed
```

Let's inspect all three with `type()` and `repr()`:

In [5]:
message = completion.choices[0].message

print("=== 1. message.content (Raw Wire JSON) ===")
print("Type:", type(message.content))
print("Raw string value:", repr(message.content))

print("\n=== 2. message.parsed (Deserialized Pydantic Object) ===")
print("Type:", type(message.parsed))
print("Is instance of CalendarEvent?:", isinstance(message.parsed, CalendarEvent))
print("Object representation:", repr(message.parsed))

print("\n=== 3. message.refusal ===")
print("Refusal status:", message.refusal)


=== 1. message.content (Raw Wire JSON) ===
Type: <class 'str'>
Raw string value: '{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}'

=== 2. message.parsed (Deserialized Pydantic Object) ===
Type: <class '__main__.CalendarEvent'>
Is instance of CalendarEvent?: True
Object representation: CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

=== 3. message.refusal ===
Refusal status: None


#### 5. Accessing Typed Attributes & Converting to Python Dict

Because `event` is a `CalendarEvent` object:
- You get typed attribute access (dot-notation) with IDE autocompletion: `event.name`, `event.date`, `event.participants`.
- `event.participants` is a genuine Python `list` of strings!
- You can convert the object to a standard Python dictionary using `event.model_dump()`.

In [6]:
event: CalendarEvent = message.parsed

print("--- Accessing Typed Attributes ---")
print(f"event.name:         {event.name} (type: {type(event.name)})")
print(f"event.date:         {event.date} (type: {type(event.date)})")
print(f"event.participants: {event.participants} (type: {type(event.participants)})")
print(f"First participant:  {event.participants[0]} (type: {type(event.participants[0])})")

print("\n--- Converting to Standard Python Dictionary (.model_dump()) ---")
event_dict = event.model_dump()
print("Type of event_dict:", type(event_dict))
print("Dictionary content:", event_dict)


--- Accessing Typed Attributes ---
event.name:         Science Fair (type: <class 'str'>)
event.date:         Friday (type: <class 'str'>)
event.participants: ['Alice', 'Bob'] (type: <class 'list'>)
First participant:  Alice (type: <class 'str'>)

--- Converting to Standard Python Dictionary (.model_dump()) ---
Type of event_dict: <class 'dict'>
Dictionary content: {'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}


#### 6. Visualizing the Complete Structured Response Object

Let's dump the entire `completion` object to inspect everything OpenAI returned, including token usage and choice metadata:

In [7]:
full_dict = completion.model_dump()

print("Full Completion Dictionary:")
print(json.dumps(full_dict, indent=2))


Full Completion Dictionary:
{
  "id": "chatcmpl-EPCKDPLr9Le2JXNU0xgieEeXtUYED",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\"name\":\"Science Fair\",\"date\":\"Friday\",\"participants\":[\"Alice\",\"Bob\"]}",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "parsed": {
          "name": "Science Fair",
          "date": "Friday",
          "participants": [
            "Alice",
            "Bob"
          ]
        }
      }
    }
  ],
  "created": 1789674285,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 414,
    "prompt_tokens": 117,
    "total_tokens": 531,
    "completion_tokens_details": {
      "accepted_predictio

c:\Users\ashut\anaconda3\envs\General_env\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CalendarEvent(name='Scien...ipants=['Alice', 'Bob']), input_type=CalendarEvent])
  return self.__pydantic_serializer__.to_python(


---

#### 7. Cheat Sheet: `model_json_schema()` vs. `model_dump()` vs. `json.loads()` vs. `.parsed`

These names blur together because they all involve "JSON" — but they move in **different directions** and belong to **different libraries**. Sort them by *what goes in* and *what comes out*.

##### The one distinction that unlocks the rest

- `model_json_schema()` describes the **shape** — it operates on the **class** and produces a *description of a type*. It contains no data. It travels **outward to the model**.
- `model_dump()` / `model_dump_json()` carry the **data** — they operate on an **instance** and produce *values*. They travel **outward to your code, a file, or another service**.

> A schema is the mould; a dump is the casting. `CalendarEvent.model_json_schema()` works without any event ever existing, whereas `event.model_dump()` needs a real `event`.

##### Full reference table

| Call | Input → Output | Library | Where it appears here |
|---|---|---|---|
| `CalendarEvent.model_json_schema()` | **class** → `dict` describing the *shape* | Pydantic | Section 2 — sent by the SDK to OpenAI to constrain decoding |
| `event.model_dump()` | **instance** → Python `dict` (values stay Python objects) | Pydantic | Section 5 — `{'name': 'Science Fair', ...}` |
| `event.model_dump_json()` | **instance** → JSON **`str`** (skips the dict step) | Pydantic | not used here; ≈ `json.dumps(event.model_dump())` |
| `json.loads(s)` | JSON **`str`** → `dict` / `list` | stdlib `json` | the manual path you'd need after `create()` |
| `json.dumps(obj)` | `dict` → JSON **`str`** | stdlib `json` | Sections 2 and 6 — pretty-printing only |
| `message.content` | *(not a call)* the raw JSON **`str`** the model emitted | OpenAI SDK field | Section 4 |
| `message.parsed` | *(not a call)* the ready-made `CalendarEvent` **instance** | OpenAI SDK, via `parse()` | Sections 4 and 5 — the payoff |
| `response.json()` | HTTP response body → `dict` | **`requests` / `httpx`** — *a different library entirely* | never in your code; runs **inside** the SDK |

##### On `response.json()`

If you have seen this elsewhere and mentally filed it with the others, separate it now. It belongs to HTTP clients like `requests` and `httpx`, and means *"parse this HTTP response body as JSON"*. It is roughly `json.loads(response.text)` plus encoding handling. It has nothing to do with Pydantic.

You never write it yourself when using the OpenAI SDK — but it **does** run, one layer down: the SDK calls httpx's `response.json()` internally to turn the raw HTTP body into a dict, before building the typed response object from that dict. `Supplement_1-basic.ipynb`, Section 4 traces that exact step. So the rule is *"not in your code"*, not *"never happens"*.

##### Why Section 6 also calls `.model_dump()`

A neat reinforcement: in Section 6 we call `completion.model_dump()` on the **response object**, not on our own event. That works because the OpenAI SDK's own response classes (`ChatCompletion`, `ParsedChatCompletion`, …) are *themselves* Pydantic `BaseModel` subclasses. Same method, same direction — instance to dict — just applied to a class the SDK authored instead of one you wrote.

##### The one-sentence version

> `model_json_schema()` sends a **shape** outward so the model cannot produce malformed output; `model_dump()` / `model_dump_json()` take a **populated object** and convert it to a dict or string for your own use; `json.loads()` is the generic, non-Pydantic way to turn a JSON string into a dict — which is exactly the step `parse()` performs on your behalf.